In [1]:
# Cell 1: Imports and Dependencies
import numpy as np
from PIL import Image
import io
import re
import subprocess
import os
from math import exp, log, sqrt
from typing import Dict, Any, Tuple, Optional
import warnings

# Check for matplotlib (required for visualization)
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
# Cell 2: FLIR Metadata Extraction Functions

def extract_flir_metadata(image_path: str, exiftool_path: str = "exiftool") -> Dict[str, Any]:
    """
    Extract FLIR radiometric metadata from image EXIF data using exiftool.
    
    Parameters
    ----------
    image_path : str
        Path to the FLIR radiometric image file (JPG/RJPG format)
    exiftool_path : str
        Path to exiftool executable (default: "exiftool" assumes it's in PATH)
    
    Returns
    -------
    dict
        Dictionary containing all relevant FLIR radiometric metadata including:
        - Planck constants (R1, R2, B, F, O)
        - Emissivity
        - Atmospheric parameters
        - Object distance
        - Temperature references
    
    Raises
    ------
    FileNotFoundError
        If the image file does not exist
    RuntimeError
        If exiftool is not available or fails to extract metadata
    ValueError
        If the image is not a valid FLIR radiometric image
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")
    
    # Define the metadata tags we need to extract
    tags = [
        '-Emissivity',
        '-ObjectDistance',
        '-ReflectedApparentTemperature',
        '-AtmosphericTemperature',
        '-IRWindowTemperature',
        '-IRWindowTransmission',
        '-RelativeHumidity',
        '-PlanckR1',
        '-PlanckR2',
        '-PlanckB',
        '-PlanckF',
        '-PlanckO',
        '-AtmosphericTransAlpha1',
        '-AtmosphericTransAlpha2',
        '-AtmosphericTransBeta1',
        '-AtmosphericTransBeta2',
        '-AtmosphericTransX',
        '-RawThermalImageWidth',
        '-RawThermalImageHeight',
        '-RawThermalImageType',
        '-DateTimeOriginal',
        '-Make',
        '-Model',
        '-j'  # JSON output
    ]
    
    try:
        cmd = [exiftool_path] + tags + [image_path]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        
        import json
        metadata_list = json.loads(result.stdout)
        
        if not metadata_list:
            raise ValueError("No metadata extracted from image")
        
        metadata = metadata_list[0]
        
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"exiftool failed: {e.stderr}")
    except FileNotFoundError:
        raise RuntimeError(
            "exiftool not found. Please install exiftool:\n"
            "  Ubuntu/Debian: sudo apt install libimage-exiftool-perl\n"
            "  macOS: brew install exiftool\n"
            "  Windows: Download from https://exiftool.org/"
        )
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Failed to parse exiftool output: {e}")
    
    return metadata


def parse_flir_metadata(metadata: Dict[str, Any]) -> Dict[str, float]:
    """
    Parse and validate FLIR metadata, extracting numeric values.
    
    Parameters
    ----------
    metadata : dict
        Raw metadata dictionary from exiftool
    
    Returns
    -------
    dict
        Parsed metadata with numeric values for all radiometric parameters
    
    Raises
    ------
    ValueError
        If required Planck constants are missing (image is not radiometric)
    """
    def extract_float(value: Any) -> Optional[float]:
        """Extract float from string or return float directly."""
        if value is None:
            return None
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            # Handle strings like "20.0 C" or "1.0 m"
            match = re.search(r'[-+]?\d*\.?\d+', str(value))
            if match:
                return float(match.group())
        return None
    
    # Check for required Planck constants
    required_planck = ['PlanckR1', 'PlanckB', 'PlanckF', 'PlanckO', 'PlanckR2']
    missing = [p for p in required_planck if p not in metadata or metadata[p] is None]
    
    if missing:
        raise ValueError(
            f"Image is not a valid FLIR radiometric image. "
            f"Missing Planck constants: {missing}\n"
            "Ensure you are using an original FLIR radiometric JPG file."
        )
    
    # Parse all parameters with defaults
    parsed = {
        # Planck constants (required)
        'PlanckR1': extract_float(metadata.get('PlanckR1')),
        'PlanckR2': extract_float(metadata.get('PlanckR2')),
        'PlanckB': extract_float(metadata.get('PlanckB')),
        'PlanckF': extract_float(metadata.get('PlanckF')),
        'PlanckO': extract_float(metadata.get('PlanckO')),
        
        # Emissivity and object parameters
        'Emissivity': extract_float(metadata.get('Emissivity')) or 0.95,
        'ObjectDistance': extract_float(metadata.get('ObjectDistance')) or 1.0,
        
        # Temperature references (in Celsius)
        'ReflectedApparentTemperature': extract_float(metadata.get('ReflectedApparentTemperature')) or 20.0,
        'AtmosphericTemperature': extract_float(metadata.get('AtmosphericTemperature')) or 20.0,
        'IRWindowTemperature': extract_float(metadata.get('IRWindowTemperature')) or 20.0,
        
        # Transmission and humidity
        'IRWindowTransmission': extract_float(metadata.get('IRWindowTransmission')) or 1.0,
        'RelativeHumidity': extract_float(metadata.get('RelativeHumidity')) or 50.0,
        
        # Atmospheric transmission constants (defaults from Thermimage)
        'AtmosphericTransAlpha1': extract_float(metadata.get('AtmosphericTransAlpha1')) or 0.006569,
        'AtmosphericTransAlpha2': extract_float(metadata.get('AtmosphericTransAlpha2')) or 0.012620,
        'AtmosphericTransBeta1': extract_float(metadata.get('AtmosphericTransBeta1')) or -0.002276,
        'AtmosphericTransBeta2': extract_float(metadata.get('AtmosphericTransBeta2')) or -0.006670,
        'AtmosphericTransX': extract_float(metadata.get('AtmosphericTransX')) or 1.9,
        
        # Image dimensions
        'RawThermalImageWidth': extract_float(metadata.get('RawThermalImageWidth')),
        'RawThermalImageHeight': extract_float(metadata.get('RawThermalImageHeight')),
        
        # Camera info
        'Make': metadata.get('Make', 'Unknown'),
        'Model': metadata.get('Model', 'Unknown'),
        'DateTimeOriginal': metadata.get('DateTimeOriginal', 'Unknown'),
    }
    
    return parsed